# Message Passing Algorithms for Scalable Multitarget Tracking test

## 基于置信传播的多目标跟踪：Python / Jupyter 实验

**本项目参考以下论文的实验代码进行 Python / Jupyter 转换与整理：**

> Meyer F, Braca P, Willett P, Hlawatsch F. A Scalable Algorithm for Tracking an Unknown Number of Targets Using Multiple Sensors[J]. IEEE Transactions on Signal Processing, 2017, 65(13): 3478–3493. DOI: 10.1109/TSP.2017.2688966.

[论文 DOI](https://doi.org/10.1109/TSP.2017.2688966) · [Florian Meyer 的原始 MATLAB 代码](https://github.com/meyer-ucsd/MTT-PIEEE-18)

原始代码作者为 **Florian Meyer（2017、2020）**，源提交为 `d2e02df90d85aca5ad1120d80f8ba66946e903ca`。该仓库的 `main.m` 同时引用2017年论文与2018年综述；本项目的名称沿用2018年综述标题，但主要实验参考文献为上述2017年论文。

本笔记本将原仓库的14个 MATLAB 文件转换为 Python / NumPy。全部算法代码均内嵌于本文件，**无需 MATLAB，也不依赖相邻的 `.m` 或 `.py` 文件**。这是学习和实验用途的非官方移植。

**VS Code：**安装 Python 与 Jupyter 扩展，选择 Python 内核后点击“全部运行”。如缺少依赖，执行下面的安装单元格并重启内核。

默认设置：200帧、每帧1秒、5个真实目标、2个传感器、每个潜在目标10000个粒子、随机种子1。没有复现论文全部对比算法或多随机种子统计；Python 与 MATLAB 随机流不同。


In [ ]:
# 仅在当前 Python 内核缺少依赖时取消下一行注释：
# %pip install numpy scipy matplotlib pandas ipykernel

## 1. 导入依赖与坐标约定

状态为 `[x, y, vx, vy]`，位置单位 m，速度单位 m/s；每个时间步为 1 s。
量测为 `[距离, 方位角]`。**方位角使用度数，从正 y 轴开始**：
`atan2(dx, dy)`；还原位置时 `x = sx + r sin(b)`、`y = sy + r cos(b)`。
为便于逐文件比对，粒子数组保持 MATLAB 的 `(4, 粒子数, 潜在目标数)` 布局。

In [ ]:
from dataclasses import dataclass, asdict
from time import perf_counter
from pathlib import Path
import warnings
import json
import importlib.metadata
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

print('Python:', sys.version.split()[0])
for package in ['numpy', 'scipy', 'matplotlib', 'pandas']:
    print(f'{package}: {importlib.metadata.version(package)}')

# Consistent plotting colors for truth, estimates, sensors and measurements.
COLORS = {'truth': '#222222', 'estimate': '#D55E00',
          'sensor': '#0072B2', 'measurement': '#56B4E9'}
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 9,
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': False,
    'legend.frameon': False, 'lines.linewidth': 1.2, 'figure.dpi': 120,
    'svg.fonttype': 'none', 'pdf.fonttype': 42})

## 2. 运动模型、初始目标和传感器

对应 `getStartStates.m`、`getSensorPositions.m`、`generateTrueTracks.m` 中的转移矩阵。
目标采用匀速状态转移，加入二维加速度噪声。五个目标从圆周向中心运动，依次出现和消失。

In [ ]:
def transition_matrices(scan_time):
    A = np.eye(4)
    A[0, 2] = A[1, 3] = scan_time
    W = np.array([[0.5*scan_time**2, 0.], [0., 0.5*scan_time**2],
                  [scan_time, 0.], [0., scan_time]])
    return A, W


def get_sensor_positions(num_sensors, radius):
    angles = np.arange(num_sensors) * 2*np.pi/num_sensors
    return radius * np.vstack([np.sin(angles), np.cos(angles)])


def get_start_states(num_targets, radius, speed):
    if num_targets == 1:
        return np.array([[0.], [0.], [speed], [0.]])
    angles = np.arange(num_targets) * 2*np.pi/num_targets
    return np.vstack([radius*np.sin(angles), radius*np.cos(angles),
                      -speed*np.sin(angles), -speed*np.cos(angles)])


def wrap_to_180(angle):
    angle = np.asarray(angle)
    wrapped = (angle + 180.) % 360. - 180.
    return np.where((wrapped == -180.) & (angle > 0), 180., wrapped)

## 3. 参数：原始实验与快速检查

默认 `QUICK_CHECK=False`，运行原始 200 步、10,000 粒子配置。
如首次仅想检查环境，可设为 `True`，运行 **30 步、500 粒子**的小实例；它不能代替原始规模结果。
`unknown_number=0.01` 保持源代码中固定均匀未知目标先验，不额外改为动态 PHD 先验。

In [ ]:
@dataclass
class Parameters:
    num_steps: int = 200
    num_particles: int = 10_000
    num_sensors: int = 2
    driving_noise_variance: float = 0.010
    survival_probability: float = 0.999
    measurement_variance_range: float = 25.0**2
    measurement_variance_bearing: float = 0.5**2
    detection_probability: float = 0.9
    measurement_range: float = 6000.0
    mean_clutter: float = 5.0
    detection_threshold: float = 0.5
    threshold_pruning: float = 1e-4
    minimum_track_length: int = 1
    unknown_number: float = 0.01
    bp_check_every: int = 20
    bp_tolerance: float = 1e-5
    bp_max_iterations: int = 100_000

    def __post_init__(self):
        if self.num_steps < 1 or self.num_particles < 1 or self.num_sensors < 1:
            raise ValueError('Steps, particles and sensors must be positive.')
        if not 0 <= self.detection_probability < 1:
            raise ValueError('This original example requires 0 <= P_D < 1.')
        if not 0 <= self.survival_probability <= 1 or self.mean_clutter <= 0:
            raise ValueError('Invalid survival probability or clutter intensity.')
        if self.bp_check_every < 1 or self.bp_max_iterations < 1:
            raise ValueError('BP iteration settings must be positive.')
        self.surveillance_region = np.array([[-3000., -3000.], [3000., 3000.]])
        self.prior_velocity_covariance = np.diag([100., 100.])
        self.length_steps = np.ones(self.num_steps)
        self.clutter_distribution = 1 / (360 * self.measurement_range)
        self.sensor_positions = get_sensor_positions(self.num_sensors, 5000.)
        self.target_start_states = get_start_states(5, 1000., 10.)
        self.target_appearance_from_to = np.array([[5, 10, 15, 20, 25],
                                                  [155, 160, 165, 170, 175]])

QUICK_CHECK = False
SEED = 1
parameters = Parameters(num_steps=30, num_particles=500) if QUICK_CHECK else Parameters()
display(pd.DataFrame({'parameter': list(asdict(parameters)), 'value': list(asdict(parameters).values())}))

## 4. 生成真实轨迹、带噪声量测、漏检和杂波

对应 `generateTrueTracks.m`、`generateTrueMeasurements.m`、`generateClutteredMeasurements.m`。
每个目标每个传感器至多生成一个量测，检测概率为 0.9；每帧每传感器杂波数量服从均值为 5 的 Poisson 分布。
目标未出现/已消失的位置用 NaN 标记，**这些 NaN 是合法缺席标记，不是计算失败**。

In [ ]:
def generate_true_tracks(p, rng):
    num_targets = p.target_start_states.shape[1]
    tracks = np.full((4, num_targets, p.num_steps), np.nan)
    for target in range(num_targets):
        state = p.target_start_states[:, target].copy()
        # MATLAB loop is 2:numSteps; step 1 is intentionally left absent.
        for k in range(1, p.num_steps):
            A, W = transition_matrices(p.length_steps[k])
            state = A @ state + W @ (np.sqrt(p.driving_noise_variance)*rng.standard_normal(2))
            start, stop = p.target_appearance_from_to[:, target]
            if start <= k + 1 <= stop:
                tracks[:, target, k] = state
    return tracks


def generate_true_measurements(target_trajectory, p, rng):
    num_targets = target_trajectory.shape[1]
    measurements = np.empty((2, num_targets, p.num_sensors))
    for sensor in range(p.num_sensors):
        delta = target_trajectory[:2] - p.sensor_positions[:, sensor, None]
        measurements[0, :, sensor] = np.hypot(*delta) + np.sqrt(p.measurement_variance_range)*rng.standard_normal(num_targets)
        # Original bearing: atan2d(delta_x, delta_y), measured from +y.
        measurements[1, :, sensor] = np.rad2deg(np.arctan2(delta[0], delta[1])) + np.sqrt(p.measurement_variance_bearing)*rng.standard_normal(num_targets)
    return measurements


def generate_cluttered_measurements(track_measurements, p, rng):
    all_measurements = []
    num_targets = track_measurements.shape[1]
    for sensor in range(p.num_sensors):
        exists = np.isfinite(track_measurements[0, :, sensor])
        detected = rng.random(num_targets) < p.detection_probability
        true_detections = track_measurements[:, exists & detected, sensor]
        n_clutter = rng.poisson(p.mean_clutter)
        clutter = np.vstack([p.measurement_range*rng.random(n_clutter),
                             360*rng.random(n_clutter)-180])
        values = np.concatenate([clutter, true_detections], axis=1)
        all_measurements.append(values[:, rng.permutation(values.shape[1])])
    return all_measurements

## 5. 粒子预测和新潜在目标

对应 `performPrediction.m` 与 `introduceNewPTs.m`（含其子函数）。
每个量测都生成一个潜在新目标；该候选是否存活由 BP 和存在概率更新决定，并非将每个量测直接判为目标。
传感器索引在 Python 内部从 0 开始，出生标签仍保留 MATLAB 的 1-based `(时间步, 传感器, 量测编号)`。

源代码 `calculateConstantsUniform` 的新生目标积分**未环绕方位角残差**，本转换保留此行为；旧目标量测似然则使用 ±180° 环绕。

In [ ]:
def perform_prediction(particles, existences, scan_time, p, rng):
    A, W = transition_matrices(scan_time)
    predicted = particles.copy()
    for target in range(particles.shape[2]):
        noise = rng.standard_normal((2, particles.shape[1]))
        predicted[:, :, target] = A @ particles[:, :, target] + W @ (np.sqrt(p.driving_noise_variance)*noise)
    return predicted, p.survival_probability*existences


def calculate_constants_uniform(sensor_position, measurements, particles, p):
    delta = particles - sensor_position[:, None]
    predicted_range = np.hypot(*delta)
    predicted_bearing = np.rad2deg(np.arctan2(delta[0], delta[1]))
    dr = measurements[0, :, None] - predicted_range
    # Deliberately no wrap here: matches calculateConstantsUniform in source.
    db = measurements[1, :, None] - predicted_bearing
    norm = 1/(2*np.pi*np.sqrt(p.measurement_variance_bearing*p.measurement_variance_range))
    likelihood = norm*np.exp(-0.5*(dr**2/p.measurement_variance_range + db**2/p.measurement_variance_bearing))
    area = np.prod(p.surveillance_region[1] - p.surveillance_region[0])
    return area*likelihood.mean(axis=1)


def sample_from_likelihood(measurement, sensor, p, rng):
    ranges = measurement[0] + np.sqrt(p.measurement_variance_range)*rng.standard_normal(p.num_particles)
    bearings = np.deg2rad(measurement[1] + np.sqrt(p.measurement_variance_bearing)*rng.standard_normal(p.num_particles))
    samples = np.empty((4, p.num_particles))
    samples[0] = p.sensor_positions[0, sensor] + ranges*np.sin(bearings)
    samples[1] = p.sensor_positions[1, sensor] + ranges*np.cos(bearings)
    # Original covariance is diagonal; Cholesky equals its principal square root.
    samples[2:] = np.linalg.cholesky(p.prior_velocity_covariance) @ rng.standard_normal((2, p.num_particles))
    return samples


def introduce_new_pts(measurements, sensor, step, unknown_particles, p, rng):
    m = measurements.shape[1]
    particles = np.empty((4, p.num_particles, m))
    labels = np.empty((3, m), dtype=int)
    area = np.prod(p.surveillance_region[1] - p.surveillance_region[0])
    unknown_intensity = p.unknown_number/area*(1-p.detection_probability)**sensor
    constants = calculate_constants_uniform(p.sensor_positions[:, sensor], measurements, unknown_particles, p)
    xi = 1 + constants*unknown_intensity*p.detection_probability/(p.mean_clutter*p.clutter_distribution)
    for j in range(m):
        particles[:, :, j] = sample_from_likelihood(measurements[:, j], sensor, p, rng)
        labels[:, j] = [step, sensor+1, j+1]  # Keep MATLAB's 1-based labels.
    return particles, labels, xi-1, xi

## 6. 旧目标的量测似然与 BP 数据关联

对应 `evaluateMeasurements.m` 和 `performDataAssociationBP.m`。
`beta[0]` 表示漏检/不存在分支；其余行是各量测关联分支。BP 在目标端与量测端之间迭代传递消息，
每 20 次检查对数消息变化是否小于 `1e-5`，上限为 `100000` 次。
该结构允许在关联存在歧义时保留软概率，而不是提前进行硬匹配。

In [ ]:
def evaluate_measurements(alphas, existence, measurements, p, sensor):
    m, t, n = measurements.shape[1], alphas.shape[2], p.num_particles
    v = np.zeros((m+1, t, n))
    beta = np.zeros((m+1, t))
    if t:
        v[0] = 1-p.detection_probability
        normalizer = p.detection_probability/(p.mean_clutter*p.clutter_distribution*2*np.pi*np.sqrt(p.measurement_variance_range*p.measurement_variance_bearing))
        for target in range(t):
            delta = alphas[:2, :, target] - p.sensor_positions[:, sensor, None]
            ranges = np.hypot(*delta)
            bearings = np.rad2deg(np.arctan2(delta[0], delta[1]))
            dr = measurements[0, :, None] - ranges
            db = wrap_to_180(measurements[1, :, None] - bearings)
            v[1:, target] = normalizer*np.exp(-0.5*(dr**2/p.measurement_variance_range + db**2/p.measurement_variance_bearing))
        beta = existence[None, :]*v.mean(axis=2)
        beta[0] += 1-existence
    return beta, v


def perform_data_association_bp(beta, xi, check_every=20, tolerance=1e-5, max_iterations=100_000):
    m, t = beta.shape[0]-1, beta.shape[1]
    kappas = np.ones((m, t))
    if not m or not t:
        return kappas, np.ones(m), {'iterations': 0, 'converged': True, 'residual': 0.}
    tiny = np.finfo(float).tiny
    residual, converged = np.inf, False
    for iteration in range(1, max_iterations+1):
        old = kappas
        product = kappas*beta[1:]
        denominator = beta[0]+product.sum(axis=0)-product
        messages = beta[1:]/np.maximum(denominator, tiny)
        denominator = xi[:, None]+messages.sum(axis=1, keepdims=True)-messages
        kappas = 1/np.maximum(denominator, tiny)
        if iteration % check_every == 0 or iteration == max_iterations:
            residual = float(np.max(np.abs(np.log(kappas)-np.log(old))))
            if residual < tolerance:
                converged = True
                break
    iotas = 1/(1+messages.sum(axis=1))
    if not converged:
        warnings.warn(f'BP did not converge in {iteration} iterations (residual={residual:.3g}).', RuntimeWarning)
    return kappas, iotas, {'iterations': iteration, 'converged': converged, 'residual': residual}

## 7. 系统重采样、存在概率更新与完整跟踪器

对应 `updatePTs.m`、`trackerBP.m` 和 `trackFormation.m`。
每帧先运动预测，然后**按传感器顺序**依次执行：新生候选 → 似然 → BP → 更新 → 剪枝。
上一传感器刚引入的候选会参与下一传感器更新。
最后 `sum(existence)` 给出期望目标数，存在概率大于 0.5 的候选输出状态估计。

In [ ]:
def resample_systematic(weights, rng):
    weights = np.asarray(weights, dtype=float)
    if weights.ndim != 1 or not np.all(np.isfinite(weights)) or np.any(weights < 0) or weights.sum() <= 0:
        raise ValueError('Resampling weights must be finite, nonnegative and have positive sum.')
    n = weights.size
    cumulative = np.cumsum(weights/weights.sum())
    cumulative[-1] = 1.0
    grid = np.arange(n)/n + rng.random()/n
    return np.searchsorted(cumulative, grid, side='right')


def update_pts(kappas, iotas, particles, new_particles, existence, new_existence, labels, new_labels, v, rng):
    updated, probabilities = particles.copy(), existence.copy()
    ess = []
    for target in range(particles.shape[2]):
        weights = v[0, target] + np.sum(kappas[:, target, None]*v[1:, target], axis=0)
        total = weights.sum()
        alive = existence[target]*total/particles.shape[1]
        probabilities[target] = alive/(alive+1-existence[target])
        if total > 0:
            normalized = weights/total
            ess.append(float(1/np.sum(normalized**2)))
            updated[:, :, target] = particles[:, resample_systematic(normalized, rng), target]
    new_probability = iotas*new_existence
    new_probability = new_probability/(new_probability+1)
    return (np.concatenate([updated, new_particles], axis=2),
            np.concatenate([probabilities, new_probability]),
            np.concatenate([labels, new_labels], axis=1), ess)


def tracker_bp(measurements, particles, existence, labels, unknown_particles, step, p, rng):
    particles, existence = perform_prediction(particles, existence, p.length_steps[step-1], p, rng)
    diagnostics = []
    for sensor in range(p.num_sensors):
        values = measurements[sensor]
        new, new_labels, new_existence, xi = introduce_new_pts(values, sensor, step, unknown_particles, p, rng)
        beta, v = evaluate_measurements(particles, existence, values, p, sensor)
        kappas, iotas, diag = perform_data_association_bp(beta, xi, p.bp_check_every, p.bp_tolerance, p.bp_max_iterations)
        particles, existence, labels, ess = update_pts(kappas, iotas, particles, new, existence, new_existence, labels, new_labels, v, rng)
        keep = existence >= p.threshold_pruning
        particles, existence, labels = particles[:, :, keep], existence[keep], labels[:, keep]
        if not np.all(np.isfinite(particles)) or not np.all(np.isfinite(existence)):
            raise FloatingPointError('Nonfinite particle states or existence probabilities.')
        if np.any((existence < 0) | (existence > 1)):
            raise FloatingPointError('Existence probability outside [0, 1].')
        diag.update(sensor=sensor+1, measurement_count=values.shape[1],
                    potential_targets=existence.size, mean_ess=float(np.mean(ess)) if ess else np.nan)
        diagnostics.append(diag)
    detected = existence > p.detection_threshold
    estimates = {'state': particles[:, :, detected].mean(axis=1), 'label': labels[:, detected].copy()}
    return estimates, existence.sum(), particles, existence, labels, diagnostics


def track_formation(estimates, minimum_track_length=1):
    label_set = sorted({tuple(label) for item in estimates for label in item['label'].T})
    lookup = {label: j for j, label in enumerate(label_set)}
    tracks = np.full((4, len(label_set), len(estimates)), np.nan)
    for k, item in enumerate(estimates):
        for j, label in enumerate(item['label'].T):
            tracks[:, lookup[tuple(label)], k] = item['state'][:, j]
    keep = np.isfinite(tracks[0]).sum(axis=1) >= minimum_track_length
    return tracks[:, keep], np.asarray(label_set, dtype=int).reshape(-1, 3)[keep]


def run_experiment(p=None, seed=1, progress_every=25):
    p = Parameters() if p is None else p
    rng = np.random.default_rng(seed)
    start = perf_counter()
    truth = generate_true_tracks(p, rng)
    # Fixed uniform unknown-target particles, as in main.m.
    unknown = p.surveillance_region[0, :, None] + (p.surveillance_region[1]-p.surveillance_region[0])[:, None]*rng.random((2, p.num_particles))
    particles = np.empty((4, p.num_particles, 0))
    existence, labels = np.empty(0), np.empty((3, 0), dtype=int)
    estimates, measurements, diagnostics = [], [], []
    cardinality = np.zeros(p.num_steps)
    step_times = np.zeros(p.num_steps)
    for k in range(p.num_steps):
        step_start = perf_counter()
        values = generate_cluttered_measurements(generate_true_measurements(truth[:, :, k], p, rng), p, rng)
        estimate, cardinality[k], particles, existence, labels, diag = tracker_bp(values, particles, existence, labels, unknown, k+1, p, rng)
        estimates.append(estimate)
        measurements.append(values)
        diagnostics.append(diag)
        step_times[k] = perf_counter()-step_start
        if progress_every and ((k+1) % progress_every == 0 or k+1 == p.num_steps):
            print(f'{k+1:3d}/{p.num_steps} | E[N]={cardinality[k]:.2f} | detected={estimate["state"].shape[1]} | elapsed={perf_counter()-start:.1f}s', flush=True)
    tracks, track_labels = track_formation(estimates, p.minimum_track_length)
    return {'parameters': p, 'seed': seed, 'truth': truth, 'tracks': tracks,
            'track_labels': track_labels, 'estimates': estimates, 'measurements': measurements,
            'cardinality': cardinality, 'diagnostics': diagnostics, 'step_times': step_times,
            'runtime_seconds': perf_counter()-start}

## 8. 运行前的可解析检查

检查角度、重采样、空输入和单目标 BP。单目标情况下可枚举关联分支，验证 BP 与精确边缘概率一致。
这些检查验证移植的关键数学行为；它们不意味着不同语言的随机序列相同。

In [ ]:
np.testing.assert_allclose(wrap_to_180(np.array([0., 181., -181., 360.])), [0., -179., 179., 0.])
empty_k, empty_i, empty_d = perform_data_association_bp(np.empty((1, 0)), np.empty(0))
assert empty_k.shape == (0, 0) and empty_i.size == 0

# One legacy target: exact association mass ∝ beta_j / xi_j.
beta_check = np.array([[0.4], [1.2], [0.7]])
xi_check = np.array([1.3, 1.8])
kappa_check, iota_check, diag_check = perform_data_association_bp(beta_check, xi_check)
exact_mass = beta_check[1:, 0]/xi_check
exact_z = beta_check[0, 0] + exact_mass.sum()
np.testing.assert_allclose(kappa_check[:, 0], 1/xi_check, rtol=1e-12)
np.testing.assert_allclose(iota_check, 1/(1+beta_check[1:, 0]/(exact_z-exact_mass)), rtol=1e-12)
idx = resample_systematic(np.array([0., 1., 0.]), np.random.default_rng(1))
assert np.all(idx == 1)
print('PASS: angle convention, empty input, exact one-target BP, systematic resampling.')

## 9. 执行实验

对应 `main.m` 的完整主循环。使用 `np.random.default_rng(SEED)`；同一 Python 环境中可重复运行。
**MATLAB 和 NumPy 的随机数生成及抽样次序不同，因此结果不应逐点与 MATLAB 的 `rng(1)` 相等。**

In [ ]:
result = run_experiment(parameters, seed=SEED, progress_every=25)
true_count = np.isfinite(result['truth'][0]).sum(axis=0)
detected_count = np.array([item['state'].shape[1] for item in result['estimates']])
assert np.isfinite(result['cardinality']).all()
assert all(d['converged'] for row in result['diagnostics'] for d in row)
print(f"Finished: {parameters.num_steps} steps, {parameters.num_particles:,} particles, "
      f"{result['runtime_seconds']:.2f} seconds; all BP updates converged.")

## 10. 结果表与附加 OSPA 指标

源程序主要显示轨迹和目标数量。本笔记本另外计算**仅位置的 OSPA**（阶数 `p=2`，截断距离 `c=200 m`），
用于检查漏检、虚警和定位误差；不是论文中已验证复现的数值。
当真值和估计都为空时为 0，仅一方为空时为 200 m。这里只运行一个种子，不绘制置信区间。

In [ ]:
def ospa_distance(truth_xy, estimated_xy, cutoff=200., order=2):
    """Supplementary position-only OSPA; not supplied by original MATLAB demo."""
    from scipy.optimize import linear_sum_assignment
    m, n = truth_xy.shape[1], estimated_xy.shape[1]
    if max(m, n) == 0:
        return 0.
    if min(m, n) == 0:
        return float(cutoff)
    cost = np.minimum(np.linalg.norm(truth_xy[:, :, None]-estimated_xy[:, None, :], axis=0), cutoff)**order
    rows, columns = linear_sum_assignment(cost)
    return float(((cost[rows, columns].sum()+abs(m-n)*cutoff**order)/max(m, n))**(1/order))


def result_table(result):
    import pandas as pd
    p = result['parameters']
    true_count = np.isfinite(result['truth'][0]).sum(axis=0)
    detected = np.array([item['state'].shape[1] for item in result['estimates']])
    ospa = []
    for k, item in enumerate(result['estimates']):
        present = np.isfinite(result['truth'][0, :, k])
        ospa.append(ospa_distance(result['truth'][:2, present, k], item['state'][:2]))
    return pd.DataFrame({'step': np.arange(1, p.num_steps+1), 'true_count': true_count,
                         'expected_count': result['cardinality'], 'detected_count': detected,
                         'ospa_position_m': ospa, 'runtime_seconds': result['step_times'],
                         'max_bp_iterations': [max(d['iterations'] for d in step) for step in result['diagnostics']],
                         'bp_converged': [all(d['converged'] for d in step) for step in result['diagnostics']]})

table = result_table(result)
summary = {
    'steps': parameters.num_steps,
    'particles': parameters.num_particles,
    'seed': SEED,
    'runtime_seconds': result['runtime_seconds'],
    'expected_count_MAE': float(np.mean(np.abs(table.expected_count-table.true_count))),
    'detected_count_MAE': float(np.mean(np.abs(table.detected_count-table.true_count))),
    'mean_position_OSPA_m': float(table.ospa_position_m.mean()),
    'all_BP_converged': bool(table.bp_converged.all()),
    'max_BP_iterations': int(table.max_bp_iterations.max()),
    'estimated_track_labels': result['tracks'].shape[1],
}
display(pd.DataFrame([summary]).T.rename(columns={0: 'value'}))
display(table.head(12))

## 11. 跟踪结果：轨迹与目标数量

上图叠加所有真实轨迹和带出生标签的估计轨迹；同一真目标可能因中断而对应多个估计标签。
下图显示真实数量、存在概率之和（连续值）和超过检测阈值的目标数（整数）。
这里保留原程序的空间范围 `[-3000, 3000] m`；传感器在 ±5000 m，见下一图的总览。

In [ ]:
steps = table.step.to_numpy()

def plot_tracking_overview(result, table):
    fig, (ax, count_ax) = plt.subplots(2, 1, figsize=(7.2, 7.0),
        gridspec_kw={'height_ratios': [2.4, 1]}, layout='constrained')
    for j in range(result['truth'].shape[1]):
        ax.plot(result['truth'][0, j], result['truth'][1, j], color=COLORS['truth'],
                label='True trajectories' if j == 0 else None)
    for j in range(result['tracks'].shape[1]):
        ax.plot(result['tracks'][0, j], result['tracks'][1, j], '--',
                color=COLORS['estimate'], alpha=0.85,
                label='Estimated trajectories' if j == 0 else None)
    ax.set(xlabel='x (m)', ylabel='y (m)', xlim=(-3000, 3000), ylim=(-3000, 3000))
    ax.set_aspect('equal', adjustable='box')
    ax.legend(loc='upper left')
    count_ax.step(table.step, table.true_count, where='post', color=COLORS['truth'], label='True count')
    count_ax.plot(table.step, table.expected_count, color=COLORS['estimate'], label='Expected count')
    count_ax.step(table.step, table.detected_count, where='post', color=COLORS['sensor'],
                  linestyle=':', label='Detected count')
    count_ax.set(xlabel='Time step (1 s)', ylabel='Number of objects',
                  xlim=(1, result['parameters'].num_steps), ylim=(0, max(6, table.detected_count.max()+1)))
    count_ax.legend(loc='upper center', ncol=3)
    return fig

overview_figure = plot_tracking_overview(result, table)
plt.show()

## 12. 传感器总览与单帧量测

左图显示两个传感器与真实运动区域；右图将某一帧的距离—方位角量测转换为平面坐标，
展示真实目标、BP 估计和包括杂波的量测。修改 `SNAPSHOT_STEP` 可查看其他帧。
视野外的杂波不会画在局部视图中；它们仍参与算法计算。

In [ ]:
SNAPSHOT_STEP = min(100, parameters.num_steps)

def plot_snapshot(result, step):
    if not 1 <= step <= result['parameters'].num_steps:
        raise ValueError('Snapshot step is outside the simulation.')
    p, k = result['parameters'], step-1
    fig, (scene_ax, ax) = plt.subplots(1, 2, figsize=(9.0, 4.2), layout='constrained')
    for j in range(result['truth'].shape[1]):
        scene_ax.plot(result['truth'][0, j], result['truth'][1, j], color=COLORS['truth'])
    scene_ax.scatter(*p.sensor_positions, marker='^', s=65, color=COLORS['sensor'], label='Sensors')
    for j, (x, y) in enumerate(p.sensor_positions.T):
        scene_ax.annotate(f'S{j+1}', (x, y), xytext=(7, 0), textcoords='offset points')
    scene_ax.set(xlabel='x (m)', ylabel='y (m)', xlim=(-5700, 5700), ylim=(-5700, 5700))
    scene_ax.set_aspect('equal', adjustable='box')
    scene_ax.legend(loc='upper left')
    for sensor, values in enumerate(result['measurements'][k]):
        angle = np.deg2rad(values[1])
        x = p.sensor_positions[0, sensor] + values[0]*np.sin(angle)
        y = p.sensor_positions[1, sensor] + values[0]*np.cos(angle)
        ax.scatter(x, y, s=17, alpha=0.6, color=COLORS['measurement'],
                   label='Measurements incl. clutter' if sensor == 0 else None)
    alive = np.isfinite(result['truth'][0, :, k])
    ax.scatter(*result['truth'][:2, alive, k], color=COLORS['truth'], marker='x', s=55, label='True objects')
    ax.scatter(*result['estimates'][k]['state'][:2], facecolors='none', edgecolors=COLORS['estimate'],
               marker='o', s=65, label='BP estimates')
    ax.set(xlabel='x (m)', ylabel='y (m)', title=f'Time step {step}',
           xlim=(-3000, 3000), ylim=(-3000, 3000))
    ax.set_aspect('equal', adjustable='box')
    ax.legend(loc='upper left', fontsize=7)
    return fig

snapshot_figure = plot_snapshot(result, SNAPSHOT_STEP)
plt.show()

## 13. 数值诊断

OSPA 有界于 `[0, 200] m`；BP 图中的迭代数包含每帧每传感器最多值，0 表示没有需迭代的关联。
原始模型对新目标使用的固定均匀粒子先验在空间上较稀疏，新目标确认可能有延迟。
这些曲线用于检查本次运行，而不是跨种子统计或方法优劣结论。

In [ ]:
assert table.ospa_position_m.between(0, 200+1e-10).all()
diagnostic_figure, axes = plt.subplots(2, 1, figsize=(7.2, 4.8), sharex=True, layout='constrained')
axes[0].plot(table.step, table.ospa_position_m, color=COLORS['sensor'])
axes[0].set(ylabel='Position OSPA (m)', ylim=(0, 205))
axes[1].step(table.step, table.max_bp_iterations, where='post', color=COLORS['estimate'])
axes[1].set(xlabel='Time step (1 s)', ylabel='Max BP iterations', ylim=(0, None))
plt.show()

## 14. 可选动画

将 `MAKE_ANIMATION` 改为 `True` 后执行此单元格，可以在笔记本内播放目标运动与估计。
动画每 2 帧取一帧，仅影响显示；跟踪计算仍使用所有时间步。无需 ffmpeg。

In [ ]:
MAKE_ANIMATION = False
if MAKE_ANIMATION:
    from matplotlib.animation import FuncAnimation
    animation_figure, animation_ax = plt.subplots(figsize=(6, 6))
    animation_ax.set(xlim=(-3000, 3000), ylim=(-3000, 3000), xlabel='x (m)', ylabel='y (m)')
    animation_ax.set_aspect('equal')
    truth_artist = animation_ax.scatter([], [], marker='x', color=COLORS['truth'], label='Truth')
    estimate_artist = animation_ax.scatter([], [], facecolors='none', edgecolors=COLORS['estimate'], label='BP')
    animation_ax.legend(loc='upper left')
    def animate(k):
        alive = np.isfinite(result['truth'][0, :, k])
        truth_artist.set_offsets(result['truth'][:2, alive, k].T)
        estimate_artist.set_offsets(result['estimates'][k]['state'][:2].T)
        animation_ax.set_title(f'Time step {k+1}')
        return truth_artist, estimate_artist
    movie = FuncAnimation(animation_figure, animate, frames=range(0, parameters.num_steps, 2), interval=100)
    plt.close(animation_figure)
    display(HTML(movie.to_jshtml()))

## 15. 保存本次结果

本单元格将逐帧表格、摘要、真值/估计轨迹、图和运行参数保存到当前工作目录的 `results/`。
默认固定随机种子，可修改 `SEED` 后重新运行第 9 节及后续单元格进行重复试验。
坐标图使用英文标签保证在不同系统的 Python 环境中正常显示，文字说明为中文。

In [ ]:
OUTPUT_DIR = Path('results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
table.to_csv(OUTPUT_DIR/'tracking_metrics.csv', index=False)
np.savez_compressed(OUTPUT_DIR/'tracking_arrays.npz', truth=result['truth'],
    estimated_tracks=result['tracks'], track_labels=result['track_labels'],
    expected_cardinality=result['cardinality'], sensor_positions=parameters.sensor_positions)
(OUTPUT_DIR/'summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
run_info = {'seed': SEED, 'parameters': asdict(parameters),
    'source_repository': 'https://github.com/meyer-ucsd/MTT-PIEEE-18',
    'source_commit': 'd2e02df90d85aca5ad1120d80f8ba66946e903ca',
    'original_matlab_sha256': {'evaluateMeasurements.m': 'c3db66644236da9446b56b76416d5e9affe010ad09a33d65c6a9b465eb0094d3', 'generateClutteredMeasurements.m': '98394c2bf567a054e598762a90a4c1e97e6735cb2e55d2ff7bebddfa029a83e9', 'generateTrueMeasurements.m': 'e04cd92d630fb0ce51c3b54f288b65d2901d7ab8c05265a1fe6826a8cdfe72cc', 'generateTrueTracks.m': 'abfc78db98ca8c27498f686218853731e6af2a95a8ddf556bc29b034cb366857', 'getSensorPositions.m': '8803feed9b0c4ac54427b99ff1e67aa287d2c9d1500561bb9a819765b4ae405f', 'getStartStates.m': '4183c3db11140fee8a15d6a074a05ff3011f9f4dd18ef7b72ef3ab9010051533', 'introduceNewPTs.m': 'd59f689891b9074eb7d6089897f11a03f445ebc09c040972f0c698f41ee8dd9e', 'main.m': '2fab4d1846bd939ea39dc1d8c6adebf58c6143313038dadd1d6967362f5b2c5a', 'performDataAssociationBP.m': 'ca71b4c3eb2caf8a863e0318a99a98235d523259c5456a39ac7227b0efbdb9a5', 'performPrediction.m': 'ecc375ad4df381bd890ed372acde1054d3d4e062660d95a4ebec3aacc64f24a8', 'showResults.m': '2206af3d1b2d012fdf918bb7d18203fbe6c1c24f49236848226ec997f469eedc', 'trackFormation.m': '3ecf01f618248a308f25adc94c510d31861b4e84fa1fc272a13f89ad2e29a123', 'trackerBP.m': 'd8a58b1e1e3c0339446e8479e56e4da8ca33d731b2a1154522d1454c572c74ba', 'updatePTs.m': '5fe8c7befbced08d155c7265dfb6192c2fec518be2ab740681a3a44c2a928658'},
    'dependencies': {name: importlib.metadata.version(name) for name in ['numpy','scipy','matplotlib','pandas']},
    'python': sys.version, 'runtime_seconds': result['runtime_seconds'],
    'note': 'Single-seed Python translation; not bitwise MATLAB equivalence or full paper benchmark.'}
(OUTPUT_DIR/'run_info.json').write_text(json.dumps(run_info, ensure_ascii=False, indent=2), encoding='utf-8')
for name, figure in [('tracking_overview', overview_figure), ('snapshot', snapshot_figure), ('diagnostics', diagnostic_figure)]:
    figure.savefig(OUTPUT_DIR/f'{name}.png', dpi=300)
    figure.savefig(OUTPUT_DIR/f'{name}.svg')
print('Results saved to:', OUTPUT_DIR.resolve())

## 16. 原文件对应关系与使用边界

| MATLAB 文件 | 本笔记本函数/部分 |
|---|---|
| `main.m` | 参数单元格、`run_experiment` 与实验运行 |
| `getStartStates.m`、`getSensorPositions.m` | `get_start_states`、`get_sensor_positions` |
| `generateTrueTracks.m` | `generate_true_tracks`、`transition_matrices` |
| `generateTrueMeasurements.m` | `generate_true_measurements` |
| `generateClutteredMeasurements.m` | `generate_cluttered_measurements` |
| `performPrediction.m` | `perform_prediction` |
| `introduceNewPTs.m` | `introduce_new_pts`、`calculate_constants_uniform`、`sample_from_likelihood` |
| `evaluateMeasurements.m` | `evaluate_measurements` |
| `performDataAssociationBP.m` | `perform_data_association_bp` |
| `updatePTs.m` | `update_pts`、`resample_systematic` |
| `trackerBP.m` | `tracker_bp` |
| `trackFormation.m` | `track_formation` |
| `showResults.m` | 轨迹、数量、单帧图与可选动画 |

**保留：**原始模型、默认参数、BP 方程、粒子均值估计、系统重采样、出生标签、传感器更新顺序与剪枝阈值。
**实现差异：**Python 与 MATLAB 随机序列不同；加入有限值检查、BP 收敛记录和接近零分母的数值保护；绘图改为笔记本内显示。
新生目标初始化仍按作者的似然采样实现，不额外增加极坐标 Jacobian 或更改出生模型。
OSPA、结果 CSV 和诊断是附加检查。未执行 MATLAB 引擎作同输入逐值比对，未复现论文全部实验表格。

原始 MATLAB 作者：**Florian Meyer（2017、2020）**；[原始代码仓库](https://github.com/meyer-ucsd/MTT-PIEEE-18)。

对应论文：

1. F. Meyer, P. Braca, P. Willett, F. Hlawatsch. *A Scalable Algorithm for Tracking an Unknown Number of Targets Using Multiple Sensors*. IEEE TSP 65, 3478–3493, 2017. [DOI](https://doi.org/10.1109/TSP.2017.2688966)
2. F. Meyer, T. Kropfreiter, J. L. Williams, R. A. Lau, F. Hlawatsch, P. Braca, M. Z. Win. *Message Passing Algorithms for Scalable Multitarget Tracking*. Proceedings of the IEEE 106, 221–259, 2018. [DOI](https://doi.org/10.1109/JPROC.2018.2789427)